# PolicyDelta

Semantic Privacy Policy Change Detector

Run the cells in order using **Runtime → Run all** in Google Colab.

In [ ]:
%pip -q install beautifulsoup4==4.14.3 PyMuPDF==1.26.7

## Configuration

In [ ]:
import hashlib
import importlib.util
import json
import platform
import tempfile
import random
import re
import sys
import unittest
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Literal

from bs4 import BeautifulSoup, NavigableString, Tag
import pymupdf


class PolicyDeltaError(ValueError):
    """Invalid policy data or application configuration."""


class EvidenceError(PolicyDeltaError):
    """Source evidence could not be verified."""


@dataclass(frozen=True, slots=True)
class Settings:
    seed: int = 42
    max_document_chars: int = 250_000
    max_pdf_pages: int = 200
    max_pdf_bytes: int = 25_000_000

    def __post_init__(self) -> None:
        if type(self.seed) is not int or not 0 <= self.seed < 2**32:
            raise PolicyDeltaError("Seed must be a nonnegative 32-bit integer.")
        if type(self.max_document_chars) is not int or self.max_document_chars < 1:
            raise PolicyDeltaError("Document character limit must be positive.")
        if type(self.max_pdf_pages) is not int or self.max_pdf_pages < 1:
            raise PolicyDeltaError("PDF page limit must be positive.")
        if type(self.max_pdf_bytes) is not int or self.max_pdf_bytes < 1:
            raise PolicyDeltaError("PDF file size limit must be positive.")


SETTINGS = Settings()


def initialize(settings: Settings = SETTINGS) -> dict[str, str | int]:
    if sys.version_info < (3, 10):
        raise RuntimeError("Python 3.10 or later is required.")
    random.seed(settings.seed)
    try:
        colab = importlib.util.find_spec("google.colab") is not None
    except (ImportError, ModuleNotFoundError, ValueError):
        colab = False
    return {
        "python": platform.python_version(),
        "environment": "colab" if colab else "python",
        "seed": settings.seed,
    }


RUNTIME = initialize()
print(f"PolicyDelta | Python {RUNTIME['python']} | {RUNTIME['environment']}")

## Document model

In [ ]:
@dataclass(frozen=True, slots=True)
class PolicyDocument:
    document_id: str
    role: Literal["old", "new"]
    text: str
    sha256: str

    def __post_init__(self) -> None:
        if not isinstance(self.document_id, str) or not self.document_id.strip():
            raise PolicyDeltaError("Document ID must be nonempty.")
        if self.role not in ("old", "new"):
            raise PolicyDeltaError("Document role must be 'old' or 'new'.")
        if not isinstance(self.text, str) or not self.text.strip():
            raise PolicyDeltaError("Policy text must be nonempty.")
        if self.sha256 != hashlib.sha256(self.text.encode("utf-8")).hexdigest():
            raise EvidenceError("Document checksum does not match its content.")

    @classmethod
    def from_text(
        cls,
        text: str,
        *,
        role: Literal["old", "new"],
        document_id: str | None = None,
        settings: Settings = SETTINGS,
    ) -> "PolicyDocument":
        if not isinstance(text, str):
            raise PolicyDeltaError("Policy content must be text.")
        if len(text) > settings.max_document_chars:
            raise PolicyDeltaError("Document exceeds the configured size limit.")
        digest = hashlib.sha256(text.encode("utf-8")).hexdigest()
        return cls(document_id if document_id is not None else f"{role}-{digest[:16]}",
                   role, text, digest)

    @classmethod
    def from_txt(
        cls,
        path: str | Path,
        *,
        role: Literal["old", "new"],
        document_id: str | None = None,
        settings: Settings = SETTINGS,
    ) -> "PolicyDocument":
        """Read UTF-8 policy text without changing its line endings."""
        try:
            source = Path(path)
        except (TypeError, ValueError) as exc:
            raise PolicyDeltaError("A valid text-file path is required.") from exc
        if source.suffix.lower() != ".txt":
            raise PolicyDeltaError("Only .txt files are supported.")
        try:
            with source.open(encoding="utf-8-sig", newline="") as file:
                text = file.read(settings.max_document_chars + 1)
        except (OSError, UnicodeError) as exc:
            raise PolicyDeltaError(f"Unable to read UTF-8 text from '{source.name}'.") from exc
        return cls.from_text(text, role=role, document_id=document_id, settings=settings)

    @classmethod
    def from_html(
        cls,
        html: str,
        *,
        role: Literal["old", "new"],
        document_id: str | None = None,
        settings: Settings = SETTINGS,
    ) -> "PolicyDocument":
        """Extract readable text from HTML with stable paragraph boundaries."""
        if not isinstance(html, str):
            raise PolicyDeltaError("HTML content must be text.")
        if len(html) > settings.max_document_chars * 8:
            raise PolicyDeltaError("HTML input exceeds the configured size limit.")

        soup = BeautifulSoup(html, "html.parser")
        root = soup.find("main") or soup.find("article") or soup.body or soup
        excluded = {"script", "style", "noscript", "template", "svg", "nav",
                    "header", "footer", "aside", "form"}
        chrome = {"nav", "navigation", "navbar", "menu", "sidebar", "breadcrumb",
                  "cookie-banner", "cookie-notice", "site-header", "site-footer"}
        for tag in list(root.find_all(True)):
            if tag.name is None:
                continue
            identifiers = [str(tag.get("id", "")), *tag.get("class", [])]
            if (tag.name in excluded or tag.has_attr("hidden")
                    or tag.get("aria-hidden") == "true"
                    or re.search(r"(?:display\s*:\s*none|visibility\s*:\s*hidden)",
                                 str(tag.get("style", "")), flags=re.I)
                    or tag.get("role") in {"navigation", "banner", "contentinfo", "search"}
                    or any(value.lower() in chrome for value in identifiers)):
                tag.decompose()

        paragraphs: list[str] = []
        blocks = {"p", "li", "blockquote", "pre", "dt", "dd", "h1", "h2", "h3",
                  "h4", "h5", "h6", "td", "th"}
        containers = blocks | {"div", "section", "article", "main", "ul", "ol",
                               "table", "tbody", "thead", "tr", "dl"}

        def append(text: str) -> None:
            normalized = re.sub(r"\s+", " ", text).strip()
            if normalized:
                paragraphs.append(normalized)

        def collect(tag: Tag) -> None:
            if tag.name in blocks and tag.find(["br", *containers]) is None:
                append(tag.get_text())
                return
            inline: list[str] = []
            for child in tag.children:
                if isinstance(child, NavigableString):
                    inline.append(str(child))
                elif isinstance(child, Tag) and child.name == "br":
                    append("".join(inline))
                    inline.clear()
                elif isinstance(child, Tag) and child.name in containers:
                    append("".join(inline))
                    inline.clear()
                    collect(child)
                elif isinstance(child, Tag):
                    inline.append(child.get_text())
            append("".join(inline))

        collect(root)
        return cls.from_text("\n\n".join(paragraphs), role=role,
                             document_id=document_id, settings=settings)

    @classmethod
    def from_html_file(
        cls,
        path: str | Path,
        *,
        role: Literal["old", "new"],
        document_id: str | None = None,
        settings: Settings = SETTINGS,
    ) -> "PolicyDocument":
        """Read a UTF-8 HTML file and extract its policy text."""
        try:
            source = Path(path)
        except (TypeError, ValueError) as exc:
            raise PolicyDeltaError("A valid HTML-file path is required.") from exc
        if source.suffix.lower() not in {".html", ".htm"}:
            raise PolicyDeltaError("Only .html and .htm files are supported.")
        try:
            with source.open(encoding="utf-8-sig", newline="") as file:
                html = file.read(settings.max_document_chars * 8 + 1)
        except (OSError, UnicodeError) as exc:
            raise PolicyDeltaError(f"Unable to read UTF-8 HTML from '{source.name}'.") from exc
        return cls.from_html(html, role=role, document_id=document_id,
                             settings=settings)

    @classmethod
    def from_pdf(
        cls,
        path: str | Path,
        *,
        role: Literal["old", "new"],
        document_id: str | None = None,
        settings: Settings = SETTINGS,
    ) -> "PolicyDocument":
        """Extract text from an unencrypted, text-based PDF."""
        try:
            source = Path(path)
        except (TypeError, ValueError) as exc:
            raise PolicyDeltaError("A valid PDF path is required.") from exc
        if source.suffix.lower() != ".pdf":
            raise PolicyDeltaError("Only .pdf files are supported.")
        try:
            if source.stat().st_size > settings.max_pdf_bytes:
                raise PolicyDeltaError("PDF exceeds the configured file size limit.")
            with pymupdf.open(filename=str(source), filetype="pdf") as pdf:
                if pdf.needs_pass:
                    raise PolicyDeltaError("Password-protected PDFs are not supported.")
                if pdf.page_count > settings.max_pdf_pages:
                    raise PolicyDeltaError("PDF exceeds the configured page limit.")
                pages: list[str] = []
                length = 0
                for page in pdf:
                    text = page.get_text("text", sort=True).strip()
                    if not text:
                        continue
                    length += len(text) + (2 if pages else 0)
                    if length > settings.max_document_chars:
                        raise PolicyDeltaError("Document exceeds the configured size limit.")
                    pages.append(text)
        except (OSError, pymupdf.FileDataError) as exc:
            raise PolicyDeltaError(f"Unable to read PDF '{source.name}'.") from exc
        if not pages:
            raise PolicyDeltaError("PDF contains no extractable text; image-only PDFs require OCR.")
        return cls.from_text("\n\n".join(pages), role=role,
                             document_id=document_id, settings=settings)

    def excerpt(self, start: int, end: int) -> "EvidenceSpan":
        if (type(start) is not int or type(end) is not int
                or start < 0 or end > len(self.text) or start >= end):
            raise EvidenceError("Evidence offsets are outside the source text.")
        return EvidenceSpan(self.document_id, start, end, self.text[start:end])


@dataclass(frozen=True, slots=True)
class EvidenceSpan:
    document_id: str
    start: int
    end: int
    text: str


def verify_evidence(span: EvidenceSpan, document: PolicyDocument) -> None:
    """Confirm that a cited excerpt is an exact slice of the source."""
    if (span.document_id != document.document_id
            or type(span.start) is not int
            or type(span.end) is not int
            or span.start < 0
            or span.start >= span.end
            or span.end > len(document.text)
            or document.text[span.start:span.end] != span.text):
        raise EvidenceError("Evidence does not match the source document.")
    if hashlib.sha256(document.text.encode("utf-8")).hexdigest() != document.sha256:
        raise EvidenceError("Source document integrity check failed.")


def document_json(document: PolicyDocument) -> str:
    return json.dumps(asdict(document), ensure_ascii=False, indent=2)

## Validation tests

In [ ]:
class PolicyDeltaTests(unittest.TestCase):
    def setUp(self) -> None:
        self.document = PolicyDocument.from_text(
            "We retain personal data for 30 days.", role="old", document_id="sample"
        )

    def test_default_settings(self):
        self.assertEqual(Settings().seed, 42)

    def test_invalid_settings(self):
        for values in ({"seed": -1}, {"seed": True}, {"max_document_chars": 0},
                       {"max_pdf_pages": 0}, {"max_pdf_bytes": 0}):
            with self.subTest(values=values), self.assertRaises(PolicyDeltaError):
                Settings(**values)

    def test_document_checksum(self):
        expected = hashlib.sha256(self.document.text.encode("utf-8")).hexdigest()
        self.assertEqual(self.document.sha256, expected)

    def test_empty_document(self):
        with self.assertRaises(PolicyDeltaError):
            PolicyDocument.from_text("  ", role="old", document_id="empty")

    def test_input_limit(self):
        with self.assertRaises(PolicyDeltaError):
            PolicyDocument.from_text("long text", role="old", document_id="large",
                                     settings=Settings(max_document_chars=5))

    def test_invalid_role(self):
        with self.assertRaises(PolicyDeltaError):
            PolicyDocument.from_text("Valid text", role="invalid", document_id="bad")

    def test_evidence_exact_match(self):
        start = self.document.text.index("30 days")
        span = self.document.excerpt(start, start + len("30 days"))
        verify_evidence(span, self.document)
        self.assertEqual(span.text, "30 days")

    def test_tampered_evidence(self):
        span = EvidenceSpan("sample", 0, 2, "No")
        with self.assertRaises(EvidenceError):
            verify_evidence(span, self.document)

    def test_evidence_other_document(self):
        span = EvidenceSpan("another", 0, 2, "We")
        with self.assertRaises(EvidenceError):
            verify_evidence(span, self.document)

    def test_out_of_bounds(self):
        with self.assertRaises(EvidenceError):
            self.document.excerpt(0, len(self.document.text) + 1)

    def test_unicode_offsets(self):
        doc = PolicyDocument.from_text("Adatkezelés ✓", role="new", document_id="unicode")
        span = doc.excerpt(12, 13)
        self.assertEqual(span.text, "✓")
        verify_evidence(span, doc)

    def test_json_serialization(self):
        result = json.loads(document_json(self.document))
        self.assertEqual(result["role"], "old")
        self.assertEqual(result["sha256"], self.document.sha256)

    def test_generated_document_id(self):
        first = PolicyDocument.from_text("Policy text", role="old")
        second = PolicyDocument.from_text("Policy text", role="old")
        self.assertEqual(first.document_id, second.document_id)
        self.assertNotEqual(first.document_id,
                            PolicyDocument.from_text("Policy text", role="new").document_id)

    def test_txt_ingestion_preserves_offsets(self):
        content = "Data retention: 30 days.\r\nPrivacy ✓"
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "privacy.TXT"
            path.write_bytes(content.encode("utf-8"))
            document = PolicyDocument.from_txt(path, role="new")
        self.assertEqual(document.text, content)
        start = document.text.index("Privacy")
        verify_evidence(document.excerpt(start, len(document.text)), document)

    def test_txt_utf8_bom(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "privacy.txt"
            path.write_bytes("Consent required".encode("utf-8-sig"))
            document = PolicyDocument.from_txt(str(path), role="old")
        self.assertEqual(document.text, "Consent required")

    def test_txt_invalid_encoding(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "privacy.txt"
            path.write_bytes(b"\xff\xfe")
            with self.assertRaises(PolicyDeltaError):
                PolicyDocument.from_txt(path, role="old")

    def test_txt_missing_or_unsupported_file(self):
        with tempfile.TemporaryDirectory() as directory:
            for name in ("missing.txt", "policy.pdf"):
                with self.subTest(name=name), self.assertRaises(PolicyDeltaError):
                    PolicyDocument.from_txt(Path(directory) / name, role="old")

    def test_txt_character_limit(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "privacy.txt"
            path.write_text("123456", encoding="utf-8")
            with self.assertRaises(PolicyDeltaError):
                PolicyDocument.from_txt(path, role="new",
                                        settings=Settings(max_document_chars=5))

    def test_txt_rejects_invalid_path(self):
        with self.assertRaises(PolicyDeltaError):
            PolicyDocument.from_txt(None, role="old")

    def test_html_extraction_and_evidence(self):
        markup = ("<html><head><title>Website</title><script>tracker()</script></head>"
                  "<body><nav>Sign in</nav><main><h1>Privacy Policy</h1>"
                  "<p>We <strong>do not</strong> share personal data.</p>"
                  "<p>Records are kept for 30 days.</p></main>"
                  "<footer>Contact us</footer></body></html>")
        document = PolicyDocument.from_html(markup, role="new")
        self.assertEqual(document.text, "Privacy Policy\n\nWe do not share personal data."
                                          "\n\nRecords are kept for 30 days.")
        start = document.text.index("do not")
        verify_evidence(document.excerpt(start, start + 6), document)

    def test_html_without_semantic_main(self):
        markup = ("<div class='navigation'>Ignore navigation</div>"
                  "<div class='policy'><div>Collected data includes email.</div>"
                  "<div>We delete it on request.</div></div>")
        document = PolicyDocument.from_html(markup, role="old")
        self.assertEqual(document.text, "Collected data includes email.\n\n"
                                        "We delete it on request.")

    def test_html_lists_and_entities(self):
        markup = "<main><h2>Your rights</h2><ul><li>Access &amp; correction</li>" \
                 "<li>Delete your data</li></ul></main>"
        document = PolicyDocument.from_html(markup, role="old")
        self.assertEqual(document.text, "Your rights\n\nAccess & correction\n\n"
                                        "Delete your data")

    def test_html_nested_lists_and_line_breaks(self):
        markup = ("<main><ul><li>Data requests<ul><li>Deletion</li></ul></li></ul>"
                  "<p>Contact us<br>for assistance</p>"
                  "<p style='display:none'>Hidden text</p></main>")
        doc = PolicyDocument.from_html(markup, role="new")
        self.assertEqual(doc.text, "Data requests\n\nDeletion\n\nContact us\n\n"
                                   "for assistance")

    def test_html_file_ingestion(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "privacy.HTM"
            path.write_bytes("<main><p>Privacy ✓</p></main>".encode("utf-8-sig"))
            document = PolicyDocument.from_html_file(path, role="new")
        self.assertEqual(document.text, "Privacy ✓")

    def test_html_malformed_and_empty(self):
        document = PolicyDocument.from_html("<main><p>Data is retained", role="new")
        self.assertEqual(document.text, "Data is retained")
        with self.assertRaises(PolicyDeltaError):
            PolicyDocument.from_html("<script>no policy text</script>", role="old")

    def test_html_invalid_files(self):
        with tempfile.TemporaryDirectory() as directory:
            missing = Path(directory) / "missing.html"
            wrong = Path(directory) / "policy.txt"
            bad = Path(directory) / "invalid.html"
            bad.write_bytes(b"\xff\xfe")
            for path in (missing, wrong, bad):
                with self.subTest(path=path.name), self.assertRaises(PolicyDeltaError):
                    PolicyDocument.from_html_file(path, role="old")

    def test_html_size_and_type_checks(self):
        for html in (None, b"<p>Text</p>"):
            with self.subTest(html=html), self.assertRaises(PolicyDeltaError):
                PolicyDocument.from_html(html, role="old")
        with self.assertRaises(PolicyDeltaError):
            PolicyDocument.from_html("<p>too much text</p>", role="old",
                                     settings=Settings(max_document_chars=3))
        with self.assertRaises(PolicyDeltaError):
            PolicyDocument.from_html("<p>123456</p>", role="old",
                                     settings=Settings(max_document_chars=5))

    def test_pdf_ingestion_and_evidence(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "policy.PDF"
            with pymupdf.open() as pdf:
                page = pdf.new_page()
                page.insert_text((72, 72), "We retain records for 90 days.")
                pdf.save(str(path))
            document = PolicyDocument.from_pdf(path, role="new")
        self.assertIn("We retain records for 90 days.", document.text)
        start = document.text.index("90 days")
        verify_evidence(document.excerpt(start, start + len("90 days")), document)

    def test_pdf_page_order_and_empty_pages(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "policy.pdf"
            with pymupdf.open() as pdf:
                pdf.new_page().insert_text((72, 72), "Data collection")
                pdf.new_page()
                pdf.new_page().insert_text((72, 72), "Data deletion")
                pdf.save(str(path))
            document = PolicyDocument.from_pdf(path, role="old")
        self.assertEqual(document.text, "Data collection\n\nData deletion")

    def test_pdf_rejects_image_only_document(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "scan.pdf"
            with pymupdf.open() as pdf:
                pdf.new_page()
                pdf.save(str(path))
            with self.assertRaisesRegex(PolicyDeltaError, "extractable text"):
                PolicyDocument.from_pdf(path, role="old")

    def test_pdf_rejects_encrypted_document(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "protected.pdf"
            with pymupdf.open() as pdf:
                pdf.new_page().insert_text((72, 72), "Restricted")
                pdf.save(str(path), encryption=pymupdf.PDF_ENCRYPT_AES_256,
                         user_pw="secret", owner_pw="owner")
            with self.assertRaisesRegex(PolicyDeltaError, "Password-protected"):
                PolicyDocument.from_pdf(path, role="old")

    def test_pdf_invalid_paths_and_data(self):
        with tempfile.TemporaryDirectory() as directory:
            corrupt = Path(directory) / "corrupt.pdf"
            corrupt.write_bytes(b"not a PDF")
            for path in (None, Path(directory) / "missing.pdf",
                         Path(directory) / "policy.txt", corrupt):
                with self.subTest(path=path), self.assertRaises(PolicyDeltaError):
                    PolicyDocument.from_pdf(path, role="old")

    def test_pdf_resource_limits(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "policy.pdf"
            with pymupdf.open() as pdf:
                pdf.new_page().insert_text((72, 72), "Data retention is 180 days")
                pdf.new_page().insert_text((72, 72), "Data deletion")
                pdf.save(str(path))
            for settings in (
                Settings(max_pdf_pages=1),
                Settings(max_pdf_bytes=10),
                Settings(max_document_chars=5),
            ):
                with self.subTest(settings=settings), self.assertRaises(PolicyDeltaError):
                    PolicyDocument.from_pdf(path, role="new", settings=settings)

    def test_reproducible_seed(self):
        initialize(Settings(seed=7))
        first = random.random()
        initialize(Settings(seed=7))
        self.assertEqual(first, random.random())
        initialize()


suite = unittest.defaultTestLoader.loadTestsFromTestCase(PolicyDeltaTests)
result = unittest.TextTestRunner(verbosity=1).run(suite)
if not result.wasSuccessful():
    raise RuntimeError("PolicyDelta validation failed.")
print(f"Validation passed: {result.testsRun} tests")